In [ ]:
!pip install -q pandas numpy scikit-learn tensorflow nltk

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving CEAS_08_cleaned.csv to CEAS_08_cleaned.csv
Saving machinewars_filtered_emails.json to machinewars_filtered_emails.json
Saving Nazario_cleaned.csv to Nazario_cleaned.csv
Saving Nigerian_Fraud_cleaned.csv to Nigerian_Fraud_cleaned.csv
Saving SpamAssasin_cleaned.csv to SpamAssasin_cleaned.csv


In [ ]:
import pandas as pd
import json
from pathlib import Path


# ---------------------------------
# 1. Helpers
# ---------------------------------
def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def build_text_subject_body(row):
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    return f"{subject}\n\n{body}".strip()


# ---------------------------------
# 2. Label handling
# ---------------------------------
def normalize_machinewars_label(label, spam_as_phishing=False):
    """
    MachineWars:
      Phishing -> 1
      Legitimate/Valid/Ham -> 0
      Spam -> 1 if spam_as_phishing=True else excluded
    """
    if pd.isna(label):
        return None

    label = str(label).strip().lower()

    if label == "phishing":
        return 1

    if label in {"legitimate", "valid", "ham", "benign", "safe"}:
        return 0

    if label == "spam":
        return 1 if spam_as_phishing else None

    return None


def normalize_test_label(label):
    """
    For CEAS-style test sets.
    Spam is excluded here unless you explicitly want otherwise.
    """
    if pd.isna(label):
        return None

    label = str(label).strip().lower()

    if label == "phishing":
        return 1

    if label in {"legitimate", "valid", "ham", "benign", "safe"}:
        return 0

    return None


# ---------------------------------
# 3. MachineWars loader
# ---------------------------------
def load_machinewars(json_path_or_list, spam_as_phishing=False, dataset_name="machinewars"):
    """
    Expected MachineWars fields:
      sender, subject, body, type, url
    """
    if isinstance(json_path_or_list, (str, Path)):
        with open(json_path_or_list, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        data = json_path_or_list

    df = pd.DataFrame(data).copy()

    # rename type -> label
    if "type" in df.columns:
        df = df.rename(columns={"type": "label"})

    # ensure required columns exist
    for col in ["subject", "body", "label"]:
        if col not in df.columns:
            df[col] = ""

    # keep optional columns if they exist
    if "sender" not in df.columns:
        df["sender"] = ""

    if "url" not in df.columns:
        df["url"] = ""

    # flatten url list to string
    df["url"] = df["url"].apply(
        lambda x: " | ".join(x) if isinstance(x, list) else safe_str(x)
    )

    df["dataset"] = dataset_name
    df["label_raw"] = df["label"].astype(str).str.strip().str.lower()
    df["label_id"] = df["label_raw"].apply(
        lambda x: normalize_machinewars_label(x, spam_as_phishing=spam_as_phishing)
    )

    # drop excluded rows
    df = df[df["label_id"].notna()].copy()
    df["label_id"] = df["label_id"].astype(int)

    # normalized binary label name
    df["label"] = df["label_id"].map({0: "legitimate", 1: "phishing"})

    # only subject + body for model text
    df["text"] = df.apply(build_text_subject_body, axis=1)

    # final schema
    df = df[
        ["dataset", "sender", "subject", "body", "url", "label_raw", "label", "label_id", "text"]
    ]

    return df


# ---------------------------------
# 4. CEAS-style test loader
# ---------------------------------
def load_ceas_style_csv(csv_path, dataset_name=None):
    """
    Assumes CEAS-style columns similar to:
      subject, body, label
    """
    csv_path = Path(csv_path)
    if dataset_name is None:
        dataset_name = csv_path.stem

    df = pd.read_csv(csv_path).copy()

    rename_map = {
        "Subject": "subject",
        "Body": "body",
        "Label": "label",
        "type": "label",
    }
    df = df.rename(columns=rename_map)

    for col in ["subject", "body", "label"]:
        if col not in df.columns:
            df[col] = ""

    if "sender" not in df.columns:
        df["sender"] = ""

    if "url" not in df.columns:
        df["url"] = ""

    df["dataset"] = dataset_name
    df["label_raw"] = df["label"].astype(str).str.strip().str.lower()
    df["label_id"] = df["label_raw"].apply(normalize_test_label)

    # drop non-binary rows
    df = df[df["label_id"].notna()].copy()
    df["label_id"] = df["label_id"].astype(int)
    df["label"] = df["label_id"].map({0: "legitimate", 1: "phishing"})
    df["text"] = df.apply(build_text_subject_body, axis=1)

    df = df[
        ["dataset", "sender", "subject", "body", "url", "label_raw", "label", "label_id", "text"]
    ]

    return df


# ---------------------------------
# 5. Create the two MachineWars versions
# ---------------------------------
machinewars_path = "machinewars_filtered_emails.json"

# Version A: spam merged into phishing
machinewars_spam_as_phishing_df = load_machinewars(
    machinewars_path,
    spam_as_phishing=True,
    dataset_name="machinewars"
)

# Version B: spam removed
machinewars_no_spam_df = load_machinewars(
    machinewars_path,
    spam_as_phishing=False,
    dataset_name="machinewars"
)

print("MachineWars: spam merged into phishing")
print(machinewars_spam_as_phishing_df["label"].value_counts())
print(machinewars_spam_as_phishing_df.head(3))

print("\nMachineWars: spam removed")
print(machinewars_no_spam_df["label"].value_counts())
print(machinewars_no_spam_df.head(3))


# ---------------------------------
# 6. Load the 4 CEAS-style test datasets
# ---------------------------------
test_paths = [
    "CEAS_08_cleaned.csv",
    "Nazario_cleaned.csv",
    "Nigerian_Fraud_cleaned.csv",
    "SpamAssasin_cleaned.csv",
]

test_dfs = [load_ceas_style_csv(p) for p in test_paths]

for i, df in enumerate(test_dfs, 1):
    print(f"\nTest dataset {i}:")
    print(df["label"].value_counts())
    print(df.head(2))

MachineWars: spam merged into phishing
label
phishing      13200
legitimate     6600
Name: count, dtype: int64
       dataset                                             sender  \
0  machinewars      Dropbox Security <noreply@dropbox-secure.net>   
1  machinewars  Google Drive Security <security-alert@google-d...   
2  machinewars  Microsoft OneDrive Security <noreply@microsoft...   

                                             subject  \
0  Unusual Sign-in Activity Detected on Your Drop...   
1  Security Alert: New Sign-in to Your Google Dri...   
2  Important Security Notification Regarding Your...   

                                                body  \
0  Dear User,\n\nWe've detected an unusual sign-i...   
1  Google Drive Security Alert\n\nWe've noticed a...   
2  Hello sarah.smith@gmail.com,\n\nThis is an aut...   

                                                 url label_raw     label  \
0         https://dropbox-security.co/account/review  phishing  phishing   
1  https:/

In [ ]:
train_df, val_df = train_test_split(
    machinewars_no_spam_df,
    test_size=0.2,
    random_state=SEED,
    stratify=machinewars_no_spam_df["label_id"]
)

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(
    train_df[["text", "label_id"]]
    .rename(columns={"label_id": "labels"})
    .reset_index(drop=True)
)

val_ds = Dataset.from_pandas(
    val_df[["text", "label_id"]]
    .rename(columns={"label_id": "labels"})
    .reset_index(drop=True)
)

In [ ]:
def compute_binary_metrics(y_true, y_pred, y_prob):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc_auc = float("nan")

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
    }

In [ ]:
X_train = train_df["text"].astype(str).tolist()
y_train = train_df["label_id"].astype(int).values

X_val = val_df["text"].astype(str).tolist()
y_val = val_df["label_id"].astype(int).values

Logistic Regression training complete.


Validation metrics:
{'accuracy': 0.9638888888888889, 'precision': 0.970954356846473, 'recall': 0.975, 'f1': 0.972972972972973, 'roc_auc': np.float64(0.9929852502295684)}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,dataset,accuracy,precision,recall,f1,roc_auc
0,validation,0.963889,0.970954,0.975000,0.972973,0.992985
1,CEAS_08_cleaned,0.696327,0.958618,0.476193,0.636302,0.914786
2,Nazario_cleaned,0.974441,1.000000,0.974441,0.987055,NaN
3,Nigerian_Fraud_cleaned,0.960084,1.000000,0.960084,0.979636,NaN
4,SpamAssasin_cleaned,0.866242,0.915269,0.603609,0.727464,0.942532


In [ ]:
from sklearn.ensemble import RandomForestClassifier
tfidf_vectorizer = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=SEED,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(X_train_tfidf, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=300, n_jobs=-1,
                       random_state=42)

In [ ]:
def rf_predict_one(text):
    X = tfidf_vectorizer.transform([str(text)])
    prob = rf_model.predict_proba(X)[0, 1]
    pred = int(prob >= 0.5)
    return pred, float(prob)


def rf_batch_predict(texts):
    X = tfidf_vectorizer.transform([str(t) for t in texts])
    probs = rf_model.predict_proba(X)[:, 1]
    preds = (probs >= 0.5).astype(int)
    return preds, probs



def evaluate_rf(df_eval):
    y_true = df_eval["label_id"].astype(int).values
    y_pred, y_prob = rf_batch_predict(df_eval["text"].astype(str).tolist())
    return compute_binary_metrics(y_true, y_pred, y_prob)

In [ ]:
print("Validation metrics:")
print(evaluate_rf(val_df))

rf_rows = [{"dataset": "validation", **evaluate_rf(val_df)}]

for test_df in test_dfs:
    test_name = test_df["dataset"].iloc[0]
    metrics = evaluate_rf(test_df)
    rf_rows.append({"dataset": test_name, **metrics})

rf_results_df = pd.DataFrame(rf_rows)
rf_results_df

Validation metrics:
{'accuracy': 0.9738023952095808, 'precision': 0.9685672514619883, 'recall': 0.9800295857988166, 'f1': 0.9742647058823529, 'roc_auc': np.float64(0.9972316545633855)}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,dataset,accuracy,precision,recall,f1,roc_auc
0,validation,0.973802,0.968567,0.980030,0.974265,0.997232
1,CEAS_08_cleaned,0.579455,0.956522,0.257852,0.406203,0.860873
2,Nazario_cleaned,0.887540,1.000000,0.887540,0.940420,NaN
3,Nigerian_Fraud_cleaned,0.556122,1.000000,0.556122,0.714754,NaN
4,SpamAssasin_cleaned,0.714925,0.606164,0.103027,0.176119,0.862637


Validation metrics:
{'accuracy': 0.9651515151515152, 'precision': 0.9742228961334344, 'recall': 0.9734848484848485, 'f1': 0.9738537324744221, 'roc_auc': np.float64(0.9911992079889806)}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,dataset,accuracy,precision,recall,f1,roc_auc
0,validation,0.965152,0.974223,0.973485,0.973854,0.991199
1,CEAS_08_cleaned,0.862517,0.931293,0.813570,0.868460,0.926345
2,Nazario_cleaned,0.941853,1.000000,0.941853,0.970056,NaN
3,Nigerian_Fraud_cleaned,0.934574,1.000000,0.934574,0.966181,NaN
4,SpamAssasin_cleaned,0.811672,0.694757,0.647846,0.670482,0.866153


In [ ]:
import nltk
from nltk.corpus import wordnet

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
def benign_prefix_attack(text):
    prefix = (
        "Hello,\n\n"
        "This is a routine business communication regarding normal account activity. "
        "Please review the information below.\n\n"
    )
    return prefix + str(text)


def benign_suffix_attack(text):
    suffix = (
        "\n\nThank you for your attention."
        "\nThis message is part of a standard informational workflow."
        "\nBest regards."
    )
    return str(text) + suffix


def contradiction_attack(text):
    prefix = (
        "Important note: this is a legitimate and safe message. "
        "It does not request any unsafe action.\n\n"
    )
    return prefix + str(text)


def prefix_injection_attack(text):
    prefix = (
        "This is a normal and trustworthy business email. "
        "The message is legitimate, safe, and routine.\n\n"
    )
    return prefix + str(text)


PHISHING_KEYWORDS = {
    "verify", "verification", "account", "password", "login", "signin",
    "security", "alert", "urgent", "confirm", "suspend", "suspended",
    "click", "update", "reset", "limited", "immediately"
}

def keyword_deletion_attack(text, max_delete=5):
    words = str(text).split()
    new_words = []
    deleted = 0

    for w in words:
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w).lower()
        if clean in PHISHING_KEYWORDS and deleted < max_delete:
            deleted += 1
            continue
        new_words.append(w)

    return " ".join(new_words)


def get_synonyms(word):
    syns = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            s = lemma.name().replace("_", " ").strip()
            if s and s.lower() != word.lower():
                syns.add(s)
    return list(syns)


def synonym_attack(text, replace_prob=0.12, max_replacements=8, seed=42):
    rng = random.Random(seed)
    words = str(text).split()
    new_words = []
    replacements = 0

    for w in words:
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w)

        if (
            replacements < max_replacements
            and len(clean) >= 4
            and clean.isalpha()
            and rng.random() < replace_prob
        ):
            syns = get_synonyms(clean)
            syns = [s for s in syns if s.isalpha() and len(s.split()) == 1]

            if syns:
                replacement = rng.choice(syns)
                if w.istitle():
                    replacement = replacement.title()
                new_words.append(replacement)
                replacements += 1
                continue

        new_words.append(w)

    return " ".join(new_words)

Active model: logistic_regression


In [ ]:
predict_one = rf_predict_one
batch_predict = rf_batch_predict
ACTIVE_MODEL_NAME = "random_forest"

print("Active model:", ACTIVE_MODEL_NAME)

Active model: random_forest


In [ ]:
def evaluate_attack_common(df_eval, attack_name, attack_fn):
    attacked_texts = [attack_fn(t) for t in df_eval["text"].astype(str).tolist()]
    y_true = df_eval["label_id"].astype(int).values

    y_pred, y_prob = batch_predict(attacked_texts)

    return {
        "attack": attack_name,
        "n_samples": len(df_eval),
        **compute_binary_metrics(y_true, y_pred, y_prob)
    }

In [ ]:
attack_rows_val = []

attack_rows_val.append(evaluate_attack_common(val_df, "clean", lambda x: x))
attack_rows_val.append(evaluate_attack_common(val_df, "benign_prefix", benign_prefix_attack))
attack_rows_val.append(evaluate_attack_common(val_df, "benign_suffix", benign_suffix_attack))
attack_rows_val.append(evaluate_attack_common(val_df, "contradiction", contradiction_attack))
attack_rows_val.append(
    evaluate_attack_common(
        val_df,
        "synonym_attack",
        lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42)
    )
)
attack_rows_val.append(
    evaluate_attack_common(
        val_df,
        "keyword_deletion",
        lambda x: keyword_deletion_attack(x, max_delete=5)
    )
)
attack_rows_val.append(evaluate_attack_common(val_df, "prefix_injection", prefix_injection_attack))

attack_results_val_df = pd.DataFrame(attack_rows_val).sort_values("f1", ascending=False)
attack_results_val_df

,attack,n_samples,accuracy,precision,recall,f1,roc_auc
4,synonym_attack,2672,0.974551,0.972059,0.977811,0.974926,0.997059
0,clean,2672,0.973802,0.968567,0.980030,0.974265,0.997232
6,prefix_injection,2672,0.971931,0.968452,0.976331,0.972376,0.996954
5,keyword_deletion,2672,0.971931,0.971914,0.972633,0.972274,0.997118
3,contradiction,2672,0.971183,0.963636,0.980030,0.971764,0.996721
1,benign_prefix,2672,0.968937,0.960116,0.979290,0.969608,0.996709
2,benign_suffix,2672,0.965569,0.965976,0.965976,0.965976,0.996017


In [ ]:
all_attack_tables = {}

for test_df in test_dfs:
    test_name = test_df["dataset"].iloc[0]

    rows = []
    rows.append(evaluate_attack_common(test_df, "clean", lambda x: x))
    rows.append(evaluate_attack_common(test_df, "benign_prefix", benign_prefix_attack))
    rows.append(evaluate_attack_common(test_df, "benign_suffix", benign_suffix_attack))
    rows.append(evaluate_attack_common(test_df, "contradiction", contradiction_attack))
    rows.append(
        evaluate_attack_common(
            test_df,
            "synonym_attack",
            lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42)
        )
    )
    rows.append(
        evaluate_attack_common(
            test_df,
            "keyword_deletion",
            lambda x: keyword_deletion_attack(x, max_delete=5)
        )
    )
    rows.append(evaluate_attack_common(test_df, "prefix_injection", prefix_injection_attack))

    result_df = pd.DataFrame(rows).sort_values("f1", ascending=False)
    all_attack_tables[test_name] = result_df

    print(f"\n=== {ACTIVE_MODEL_NAME} | {test_name} ===")
    print(result_df)


=== random_forest | CEAS_08_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
4    synonym_attack      39154  0.580962   0.954660  0.261240  0.410223   
0             clean      39154  0.579455   0.956522  0.257852  0.406203   
5  keyword_deletion      39154  0.577131   0.958211  0.252999  0.400304   
1     benign_prefix      39154  0.549701   0.962036  0.200714  0.332134   
3     contradiction      39154  0.456863   0.907932  0.029347  0.056856   
6  prefix_injection      39154  0.449737   0.879795  0.015749  0.030945   
2     benign_suffix      39154  0.449405   0.879679  0.015063  0.029618   

    roc_auc  
4  0.856634  
0  0.860873  
5  0.866492  
1  0.859517  
3  0.849078  
6  0.846148  
2  0.854561  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist


=== random_forest | Nazario_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
1     benign_prefix       1565  0.914377        1.0  0.914377  0.955274   
3     contradiction       1565  0.893930        1.0  0.893930  0.943995   
0             clean       1565  0.887540        1.0  0.887540  0.940420   
4    synonym_attack       1565  0.872843        1.0  0.872843  0.932105   
6  prefix_injection       1565  0.861981        1.0  0.861981  0.925875   
2     benign_suffix       1565  0.829393        1.0  0.829393  0.906741   
5  keyword_deletion       1565  0.823642        1.0  0.823642  0.903294   

   roc_auc  
1      NaN  
3      NaN  
0      NaN  
4      NaN  
6      NaN  
2      NaN  
5      NaN  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist


=== random_forest | Nigerian_Fraud_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
1     benign_prefix       3332  0.602641        1.0  0.602641  0.752060   
3     contradiction       3332  0.602041        1.0  0.602041  0.751592   
6  prefix_injection       3332  0.577731        1.0  0.577731  0.732357   
2     benign_suffix       3332  0.556723        1.0  0.556723  0.715250   
0             clean       3332  0.556122        1.0  0.556122  0.714754   
4    synonym_attack       3332  0.539016        1.0  0.539016  0.700468   
5  keyword_deletion       3332  0.419568        1.0  0.419568  0.591121   

   roc_auc  
1      NaN  
3      NaN  
6      NaN  
2      NaN  
0      NaN  
4      NaN  
5      NaN  

=== random_forest | SpamAssasin_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
1     benign_prefix       5809  0.737304   0.903361  0.125146  0.219836   
4    synonym_attack       5809  0.716474   0.619529 

NameError: name 'logreg_results_df' is not defined

FileNotFoundError: [Errno 2] No such file or directory: '/content/results'

In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
def benign_prefix_attack(text):
    prefix = (
        "Hello,\n\n"
        "This is a routine business communication regarding normal account activity. "
        "Please review the information below.\n\n"
    )
    return prefix + str(text)


def benign_suffix_attack(text):
    suffix = (
        "\n\nThank you for your attention."
        "\nThis message is part of a standard informational workflow."
        "\nBest regards."
    )
    return str(text) + suffix


def contradiction_attack(text):
    prefix = (
        "Important note: this is a legitimate and safe message. "
        "It does not request any unsafe action.\n\n"
    )
    return prefix + str(text)


def training_context_attack(text):
    prefix = (
        "Example of a safe enterprise email used for routine staff communication:\n\n"
    )
    return prefix + str(text)


def noise_injection_attack(text):
    noise = (
        "\n\nReference ID: A7K2-91LM"
        "\nDepartment: operations"
        "\nCase Type: routine review"
    )
    return str(text) + noise

In [ ]:
import re

STOPWORDS_ATTACK = {
    "the", "a", "an", "and", "or", "to", "of", "in", "for", "on", "at",
    "is", "are", "this", "that", "with", "from", "by", "as", "it", "be",
    "was", "were", "subject", "body", "dear", "hello", "hi", "regards",
    "thanks", "thank", "best"
}


def basic_tokenize_with_indices(text):
    """
    Splits text into whitespace-separated tokens and keeps positions.
    """
    tokens = str(text).split()
    return tokens


def is_deletable_token(tok):
    clean = re.sub(r"^[^\w]+|[^\w]+$", "", tok).lower()
    if len(clean) < 3:
        return False
    if clean in STOPWORDS_ATTACK:
        return False
    if not any(ch.isalpha() for ch in clean):
        return False
    return True


def delete_token_at_index(tokens, idx):
    return " ".join(tokens[:idx] + tokens[idx+1:])


def greedy_delete_attack_blackbox(text, max_delete_steps=5, candidate_cap=15):
    """
    Black-box deletion attack:
    At each step, test candidate single-token deletions and keep the one that
    minimizes phishing probability.
    """
    current_text = str(text)

    for _ in range(max_delete_steps):
        tokens = basic_tokenize_with_indices(current_text)

        candidate_indices = [i for i, tok in enumerate(tokens) if is_deletable_token(tok)]

        if not candidate_indices:
            break

        # Cap candidates for speed
        candidate_indices = candidate_indices[:candidate_cap]

        candidate_texts = [delete_token_at_index(tokens, i) for i in candidate_indices]

        _, candidate_probs = batch_predict(candidate_texts)
        best_idx = int(np.argmin(candidate_probs))

        best_text = candidate_texts[best_idx]

        if best_text == current_text:
            break

        current_text = best_text

    return current_text

In [ ]:
def greedy_add_attack_blackbox(text, add_steps=3):
    """
    Greedily applies the benign addition that lowers phishing probability the most.
    """
    current_text = str(text)

    addition_fns = [
        ("benign_prefix", benign_prefix_attack),
        ("benign_suffix", benign_suffix_attack),
        ("contradiction", contradiction_attack),
        ("training_context", training_context_attack),
        ("noise_injection", noise_injection_attack),
    ]

    history = []

    for step in range(1, add_steps + 1):
        candidate_names = []
        candidate_texts = []

        for attack_name, attack_fn in addition_fns:
            candidate_names.append(attack_name)
            candidate_texts.append(attack_fn(current_text))

        candidate_preds, candidate_probs = batch_predict(candidate_texts)

        best_idx = int(np.argmin(candidate_probs))
        current_text = candidate_texts[best_idx]

        history.append({
            "step": step,
            "attack_name": candidate_names[best_idx],
            "pred": int(candidate_preds[best_idx]),
            "phishing_prob": float(candidate_probs[best_idx]),
        })

    return current_text, history

In [ ]:
def add_only_attack(text, add_steps=3):
    attacked_text, _ = greedy_add_attack_blackbox(text, add_steps=add_steps)
    return attacked_text


def delete_only_attack(text, delete_steps=5):
    attacked_text = greedy_delete_attack_blackbox(text, max_delete_steps=delete_steps)
    return attacked_text


def hybrid_add_then_delete_attack_blackbox(text, add_steps=3, delete_steps=5):
    """
    First greedy additions, then greedy deletions.
    """
    current_text, add_history = greedy_add_attack_blackbox(text, add_steps=add_steps)
    current_text = greedy_delete_attack_blackbox(current_text, max_delete_steps=delete_steps)
    return current_text

In [ ]:
def evaluate_attack_detailed(df_eval, attack_name, attack_fn, show_progress=True):
    y_true = df_eval["label_id"].astype(int).tolist()
    texts = df_eval["text"].astype(str).tolist()

    orig_pred, orig_prob = batch_predict(texts)

    attacked_texts = []
    iterator = texts
    if show_progress:
        iterator = tqdm(texts, total=len(texts), desc=f"{ACTIVE_MODEL_NAME} | {attack_name}")

    for text in iterator:
        attacked_texts.append(attack_fn(text))

    new_pred, new_prob = batch_predict(attacked_texts)

    metrics = compute_binary_metrics(y_true, new_pred, new_prob)

    details_df = pd.DataFrame({
        "text": texts,
        "label_id": y_true,
        "orig_pred": orig_pred,
        "orig_prob": orig_prob,
        "attacked_text": attacked_texts,
        "new_pred": new_pred,
        "new_prob": new_prob,
    })

    details_df["flipped"] = details_df["orig_pred"] != details_df["new_pred"]
    details_df["prob_drop"] = details_df["orig_prob"] - details_df["new_prob"]

    summary = {
        "attack": attack_name,
        "n_samples": len(df_eval),
        "flip_rate": float(details_df["flipped"].mean()),
        "avg_prob_drop": float(details_df["prob_drop"].mean()),
        **metrics,
    }

    return summary, details_df

In [ ]:
def evaluate_evasion_on_phishing(df_eval, attack_name, attack_fn, show_progress=True):
    """
    Restrict evaluation to:
    - true phishing samples
    - originally correct phishing predictions

    Reports attack success rate (ASR).
    """
    df_local = df_eval.copy()
    df_local = df_local[df_local["label_id"].astype(int) == 1].copy()

    if len(df_local) == 0:
        return {
            "attack": attack_name,
            "n_true_phishing": 0,
            "n_orig_correct_phishing": 0,
            "n_flipped": 0,
            "attack_success_rate": float("nan"),
            "robust_recall_on_orig_correct_phishing": float("nan"),
            "avg_prob_drop": float("nan"),
        }, pd.DataFrame()

    texts = df_local["text"].astype(str).tolist()
    orig_pred, orig_prob = batch_predict(texts)

    df_local["orig_pred"] = orig_pred
    df_local["orig_prob"] = orig_prob

    df_attack = df_local[df_local["orig_pred"] == 1].copy()

    if len(df_attack) == 0:
        return {
            "attack": attack_name,
            "n_true_phishing": len(df_local),
            "n_orig_correct_phishing": 0,
            "n_flipped": 0,
            "attack_success_rate": float("nan"),
            "robust_recall_on_orig_correct_phishing": float("nan"),
            "avg_prob_drop": float("nan"),
        }, pd.DataFrame()

    attacked_texts = []
    iterator = df_attack["text"].astype(str).tolist()
    if show_progress:
        iterator = tqdm(iterator, total=len(df_attack), desc=f"{ACTIVE_MODEL_NAME} | {attack_name} phishing")

    for text in iterator:
        attacked_texts.append(attack_fn(text))

    new_pred, new_prob = batch_predict(attacked_texts)

    details_df = pd.DataFrame({
        "text": df_attack["text"].astype(str).tolist(),
        "label_id": df_attack["label_id"].astype(int).tolist(),
        "orig_pred": df_attack["orig_pred"].tolist(),
        "orig_prob": df_attack["orig_prob"].tolist(),
        "attacked_text": attacked_texts,
        "new_pred": new_pred,
        "new_prob": new_prob,
    })

    details_df["flipped"] = details_df["new_pred"] != 1
    details_df["prob_drop"] = details_df["orig_prob"] - details_df["new_prob"]

    n_orig_correct = len(details_df)
    n_flipped = int(details_df["flipped"].sum())
    asr = n_flipped / n_orig_correct
    robust_recall = 1.0 - asr

    summary = {
        "attack": attack_name,
        "n_true_phishing": len(df_local),
        "n_orig_correct_phishing": n_orig_correct,
        "n_flipped": n_flipped,
        "attack_success_rate": asr,
        "robust_recall_on_orig_correct_phishing": robust_recall,
        "avg_prob_drop": float(details_df["prob_drop"].mean()),
    }

    return summary, details_df

In [ ]:
val_attack_summaries = []
val_attack_details = {}

attack_specs = [
    ("add_only_add3", lambda x: add_only_attack(x, add_steps=3)),
    ("delete_only_del5", lambda x: delete_only_attack(x, delete_steps=5)),
    ("hybrid_add3_delete5", lambda x: hybrid_add_then_delete_attack_blackbox(x, add_steps=3, delete_steps=5)),
]

In [ ]:
for attack_name, attack_fn in attack_specs:
    summary, details = evaluate_attack_detailed(
        val_df,
        attack_name,
        attack_fn,
        show_progress=True
    )
    val_attack_summaries.append(summary)
    val_attack_details[attack_name] = details

val_attack_results_df = pd.DataFrame(val_attack_summaries).sort_values("f1", ascending=False)
val_attack_results_df

random_forest | add_only_add3:   0%|          | 0/2672 [00:00<?, ?it/s]

random_forest | delete_only_del5:   0%|          | 0/2672 [00:00<?, ?it/s]

random_forest | hybrid_add3_delete5:   0%|          | 0/2672 [00:00<?, ?it/s]

,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
1,delete_only_del5,2672,0.010479,0.019497,0.973054,0.976900,0.969675,0.973274,0.996691
0,add_only_add3,2672,0.019087,0.019573,0.962201,0.973505,0.951183,0.962215,0.996066
2,hybrid_add3_delete5,2672,0.025075,0.028409,0.959955,0.977744,0.942308,0.959699,0.995879


In [ ]:
val_evasion_summaries = []
val_evasion_details = {}

for attack_name, attack_fn in attack_specs:
    summary, details = evaluate_evasion_on_phishing(
        val_df,
        attack_name,
        attack_fn,
        show_progress=True
    )
    val_evasion_summaries.append(summary)
    val_evasion_details[attack_name] = details

val_evasion_results_df = pd.DataFrame(val_evasion_summaries).sort_values("attack_success_rate", ascending=False)
val_evasion_results_df

random_forest | add_only_add3 phishing:   0%|          | 0/1325 [00:00<?, ?it/s]

random_forest | delete_only_del5 phishing:   0%|          | 0/1325 [00:00<?, ?it/s]

random_forest | hybrid_add3_delete5 phishing:   0%|          | 0/1325 [00:00<?, ?it/s]

,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
2,hybrid_add3_delete5,1352,1325,51,0.038491,0.961509,0.046179
0,add_only_add3,1352,1325,39,0.029434,0.970566,0.036420
1,delete_only_del5,1352,1325,14,0.010566,0.989434,0.025776


In [ ]:
all_test_attack_results = {}
all_test_evasion_results = {}

for test_df in test_dfs:
    dataset_name = test_df["dataset"].iloc[0]

    print(f"\n\n==============================")
    print(f"Starting dataset: {dataset_name}")
    print(f"Rows: {len(test_df)}")
    print(f"==============================")

    attack_summaries = []
    evasion_summaries = []

    for attack_name, attack_fn in attack_specs:
        print(f"\nRunning attack: {attack_name}")

        summary_attack, _ = evaluate_attack_detailed(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        attack_summaries.append({"dataset": dataset_name, **summary_attack})

        summary_evasion, _ = evaluate_evasion_on_phishing(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        evasion_summaries.append({"dataset": dataset_name, **summary_evasion})

        print("\nAttack metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_attack}]))

        print("\nPhishing evasion metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_evasion}]))

    dataset_attack_df = pd.DataFrame(attack_summaries)
    dataset_evasion_df = pd.DataFrame(evasion_summaries)

    all_test_attack_results[dataset_name] = dataset_attack_df
    all_test_evasion_results[dataset_name] = dataset_evasion_df

    print(f"\nFinished dataset: {dataset_name}")

    print("\nAll attack metrics for this dataset:")
    display(dataset_attack_df)

    print("\nAll phishing evasion metrics for this dataset:")
    display(dataset_evasion_df)



Starting dataset: CEAS_08_cleaned
Rows: 39154

Running attack: add_only_add3


random_forest | add_only_add3:   0%|          | 0/39154 [00:00<?, ?it/s]

random_forest | add_only_add3 phishing:   0%|          | 0/5632 [00:00<?, ?it/s]


Attack metrics:


,dataset,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,CEAS_08_cleaned,add_only_add3,39154,0.143408,0.083626,0.448307,0.938182,0.011812,0.02333,0.854615


...

,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,CEAS_08_cleaned,add_only_add3,21842,5632,5375,0.954368,0.045632,0.369348
1,CEAS_08_cleaned,delete_only_del5,21842,5632,332,0.058949,0.941051,0.001964
2,CEAS_08_cleaned,hybrid_add3_delete5,21842,5632,5386,0.956321,0.043679,0.383208




Starting dataset: Nazario_cleaned
Rows: 1565

Running attack: add_only_add3


random_forest | add_only_add3:   0%|          | 0/1565 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


random_forest | add_only_add3 phishing:   0%|          | 0/1389 [00:00<?, ?it/s]

In [ ]:
all_test_attack_results = {}
all_test_evasion_results = {}

for test_df in test_dfs:
    dataset_name = test_df["dataset"].iloc[0]

    if dataset_name == "CEAS_08_cleaned":
        print(f"\nSkipping dataset: {dataset_name}")
        continue

    print(f"\n\n==============================")
    print(f"Starting dataset: {dataset_name}")
    print(f"Rows: {len(test_df)}")
    print(f"==============================")

    attack_summaries = []
    evasion_summaries = []

    for attack_name, attack_fn in attack_specs:
        print(f"\nRunning attack: {attack_name}")

        summary_attack, _ = evaluate_attack_detailed(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        attack_summaries.append({"dataset": dataset_name, **summary_attack})

        summary_evasion, _ = evaluate_evasion_on_phishing(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        evasion_summaries.append({"dataset": dataset_name, **summary_evasion})

        print("\nAttack metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_attack}]))

        print("\nPhishing evasion metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_evasion}]))

    dataset_attack_df = pd.DataFrame(attack_summaries)
    dataset_evasion_df = pd.DataFrame(evasion_summaries)

    all_test_attack_results[dataset_name] = dataset_attack_df
    all_test_evasion_results[dataset_name] = dataset_evasion_df

    print(f"\nFinished dataset: {dataset_name}")

    print("\nAll attack metrics for this dataset:")
    display(dataset_attack_df)

    print("\nAll phishing evasion metrics for this dataset:")
    display(dataset_evasion_df)


Skipping dataset: CEAS_08_cleaned


Starting dataset: Nazario_cleaned
Rows: 1565

Running attack: add_only_add3


random_forest | add_only_add3:   0%|          | 0/1565 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


random_forest | add_only_add3 phishing:   0%|          | 0/1389 [00:00<?, ?it/s]


Attack metrics:


...

,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,SpamAssasin_cleaned,hybrid_add3_delete5,1718,177,144,0.813559,0.186441,0.283823



Finished dataset: SpamAssasin_cleaned

All attack metrics for this dataset:


,dataset,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,SpamAssasin_cleaned,add_only_add3,5809,0.043209,0.031223,0.711310,1.000000,0.023865,0.046617,0.909940
1,SpamAssasin_cleaned,delete_only_del5,5809,0.006542,0.012100,0.713892,0.608527,0.091385,0.158907,0.856759
2,SpamAssasin_cleaned,hybrid_add3_delete5,5809,0.044586,0.039398,0.709933,1.000000,0.019208,0.037693,0.910528



All phishing evasion metrics for this dataset:


,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,SpamAssasin_cleaned,add_only_add3,1718,177,136,0.768362,0.231638,0.270075
1,SpamAssasin_cleaned,delete_only_del5,1718,177,22,0.124294,0.875706,0.012166
2,SpamAssasin_cleaned,hybrid_add3_delete5,1718,177,144,0.813559,0.186441,0.283823


In [ ]:
combined_attack_df = pd.concat(all_test_attack_results.values(), ignore_index=True)
combined_evasion_df = pd.concat(all_test_evasion_results.values(), ignore_index=True)

print("Combined attack results:")
print(combined_attack_df)

print("\nCombined evasion results:")
print(combined_evasion_df)

import os
os.makedirs("/content/results", exist_ok=True)

val_attack_results_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_val_add_delete_hybrid_metrics.csv", index=False)
val_evasion_results_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_val_add_delete_hybrid_evasion.csv", index=False)
combined_attack_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_test_add_delete_hybrid_metrics.csv", index=False)
combined_evasion_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_test_add_delete_hybrid_evasion.csv", index=False)

print("Saved.")